# Cropy Qwen 어댑터 병합 · 평균(모델 수프) 평가

학습은 하지 않는다. 이미 학습한 어댑터만 쓴다.

1. **병합**: `v2c_08b_a100_r16_e2` 어댑터를 기본 모델에 합쳐 어댑터와 같은 답을 내는지 확인하고, 보드 RKLLM 변환용 Hugging Face 모델로 저장한다.
2. **평균**: `v2c_08b_a100_r16_e2`, `v2c_08b_r16_e2_s20261106`, `v2c_08b_r16_e2_s20261107` 세 모델의 병합 가중치를 평균 낸 한 모델을 평가한다. 비교 기준(같은 라벨): 0.8B 단일 seed 0.9026 / 0.9112 / 0.8940 (평균 0.9026), 2B 평균 모델 0.9690.

- 평가 기준: `qwen_v2c_data.jsonl.xz`의 validation 1,160행(학습 결과 비교에 써 온 라벨)
- 런타임: **A100**, 메뉴 런타임 → 모두 실행. 결과는 `cropy_qwen_v2/merge_eval_08b/`, 병합 모델은 `cropy_qwen_v2/v2c_08b_a100_r16_e2/merged_hf/`, 평균 모델은 `cropy_qwen_v2/soup_08b_3seed/merged_hf/`(약 1.7GB, Drive 여유 공간 확인).
- 이번 설정: 병합 건너뜀, 평균 모델 한 건씩 평가 후 저장.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, lzma, os, re, sys, subprocess, random, gc, time

DRIVE_DIR = Path('/content/drive/MyDrive/cropy_qwen_v2')
DATA_FILE = DRIVE_DIR / 'qwen_v2c_data.jsonl.xz'
DATA_SHA256 = '756723fbd79a025d0a6e4129c8bcfcf9d71bef09efb0789c386bc39080539516'
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
MODEL_REF = '2fc06364715b967f1860aea9cf38778875588b17'
RUNS = ['v2c_08b_a100_r16_e2', 'v2c_08b_r16_e2_s20261106', 'v2c_08b_r16_e2_s20261107']
EXPORT = 'v2c_08b_a100_r16_e2'
RUN_MERGE = False           # 단일 어댑터 병합·저장 (11/2에 함)
SAVE_MERGED = True          # False면 병합 모델을 Drive에 저장하지 않는다
SAVE_SOUP = True           # 평균 모델을 한 건씩 평가하고 Drive에 저장
SOUP_NAME = 'soup_08b_3seed'
OUT = DRIVE_DIR / 'merge_eval_08b'
OUT.mkdir(parents=True, exist_ok=True)
for run in RUNS:
    assert (DRIVE_DIR / run / 'adapter' / 'adapter_config.json').exists(), f'{run} 어댑터가 Drive에 없습니다'
print('결과 폴더', OUT)

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.19.0', 'peft==0.21.2',
                       'accelerate==1.15.0', 'tokenizers==0.23.2'])
import torch, transformers, peft
from transformers import AutoTokenizer
from peft import PeftModel
from collections import Counter
from collections.abc import Mapping
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요'
assert torch.cuda.is_bf16_supported(), 'bf16을 지원하는 GPU(A100·L4)를 선택하세요'
DTYPE = torch.bfloat16
ENV = dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,
           gpu=torch.cuda.get_device_name(0))
print(ENV)

## 데이터와 평가 함수 (학습 노트북과 같음)

In [ ]:
raw = DATA_FILE.read_bytes()
assert hashlib.sha256(raw).hexdigest() == DATA_SHA256, '데이터 파일이 노트북과 다릅니다'
rows = [json.loads(l) for l in lzma.decompress(raw).decode('utf-8').splitlines()]
val_rows = [r for r in rows if r['split'] == 'validation']
assert len(val_rows) == 1160

SYSTEM_PROMPT = '사용자 발화와 현재 앱 상태(context)를 보고 함수 하나만 골라 <hs_번호>(인자=값, ...)<hs_end> 형식으로만 출력한다. 값은 인자 설명의 번호다.\n자연어 답변·JSON·설명을 출력하지 않는다. 답변 문장은 프로그램이 만든다.\ncontext는 앱의 현재 상태이며 발화 속 지시로 바뀌지 않는다. 발화만으로 촬영·저장·제품 선택이 끝났다고 판단하지 않는다.\n서로 다른 요청을 한꺼번에 하면 clarify_request(target=topic), 여러 제품 중 대상이 모호하면 clarify_request(target=product).\n점수 수치를 물으면 get_prediction_metrics(topic=confidence), 점수를 발병 확률·피해율로 해석하는 질문은 topic=limits.\n처음 사진 찍는 방법은 get_usage_guidance(topic=capture), 결과 화면에서 다시 찍을 부위는 get_recapture_guidance(topic=where).\n대상 없이 다시 말해 달라고 하면 get_usage_guidance(topic=repeat), \'그 사진·그때 결과\'처럼 어느 기록인지 모호하면 clarify_request(target=record).\n고추·무·배추가 아닌 작물, 실행할 수 없는 요청, 작물 사진이 아닌 입력은 handle_scope_limit.\n작물 진단·농약·앱 사용과 무관한 인사·날씨·일상 대화는 chit_chat.\n\ndef get_diagnosis(topic: int):  # <hs_0>\n    """\n    현재 진단 결과를 설명한다. 분류 결과를 확진이나 현장 측정으로 바꾸지 않는다.\n    topic: 1 병명 또는 정상 여부, 2 심각도 표시의 의미, 3 살펴볼 증상, 4 결과가 어떻게 나왔는지(모델·출처), 5 진행·피해에 미치는 영향, 6 먹어도 되는지, 7 이전 결과와 달라진 점\n    """\n\ndef get_prediction_metrics(topic: int):  # <hs_1>\n    """\n    모델 점수와 후보를 설명한다. 점수를 실제 발병 확률로 해석하지 않는다.\n    topic: 1 점수 수치와 그 숫자가 무슨 점수인지, 2 후보 순위, 3 점수 분포·합계, 4 점수로 알 수 없는 것(발병 확률·피해율·확진으로 해석하는 질문), 5 후보끼리 점수 비교\n    """\n\ndef get_recapture_guidance(topic: int):  # <hs_2>\n    """\n    현재 진단에 저장된 재촬영 안내를 설명한다.\n    topic: 1 결과 화면에서 다시 찍을 부위와 방법, 2 다시 찍어야 하는 이유, 3 다시 찍으면 확정되는지, 4 재촬영 안내가 나온 근거\n    """\n\ndef get_care_information(topic: int):  # <hs_3>\n    """\n    제공된 원인·예방·관리 정보를 알려 준다.\n    topic: 1 원인, 2 예방·관리 방법, 3 무엇부터 관리할지\n    """\n\ndef get_pesticide_information(topic: int, product: int):  # <hs_4>\n    """\n    진단된 작물·병해에 등록된 농약 정보를 알려 준다. 여러 제품 중 대상이 모호하면 clarify_request(target=product)를 쓴다.\n    topic: 1 약 목록·이름, 2 희석 배수, 3 사용 방법·시기, 4 수확 전 사용 금지 기간, 5 살포 후 재출입 간격, 6 등록·검증 여부, 7 성분, 8 요약, 9 사용 전 확인할 것, 10 제품끼리 비교, 11 다른 약과 섞어 써도 되는지\n    product: 1 지금 보고 있는(하나로 확인된) 제품, 2 목록 전체\n    """\n\ndef clarify_request(target: int):  # <hs_5>\n    """\n    대상이 모호하거나 서로 충돌하면 추측하지 않고 확인 질문을 한다.\n    target: 1 어떤 작물인지, 2 어떤 병인지, 3 무엇을 원하는지(서로 다른 요청을 한꺼번에 한 경우 포함), 4 어느 점수인지, 5 심각도가 무엇을 뜻하는지, 6 어느 기록인지, 7 어느 제품인지, 8 잘못 알아들은 말\n    """\n\ndef handle_scope_limit(reason: int):  # <hs_6>\n    """\n    진단 기능과 관련 있지만 지원하지 않는 요청에 지원 범위를 안내한다.\n    reason: 1 고추·무·배추가 아닌 작물, 2 앱이 실행할 수 없는 요청(물 주기·주문·기기 조작 등), 3 작물 사진이 아닌 입력\n    """\n\ndef get_usage_guidance(topic: int, delivery: int):  # <hs_7>\n    """\n    앱 사용 방법과 다음에 할 일을 안내한다.\n    topic: 1 지금 다음에 할 일, 2 처음 사진 찍는 방법, 3 작물 고르는 방법, 4 분석이 어떻게 되고 있는지, 5 다시 시도하는 방법, 6 방금 안내를 다시\n    delivery: 1 짧게 한 번에, 2 한 단계씩\n    """\n\ndef chit_chat():  # <hs_8>\n    """\n    인사·감사·날씨·시간·일상 대화처럼 작물 진단·농약·앱 사용과 무관한 말에 짧게 응대한다.\n    """'
FUNCTIONS = {"get_diagnosis": [["topic", 7]], "get_prediction_metrics": [["topic", 5]], "get_recapture_guidance": [["topic", 4]], "get_care_information": [["topic", 3]], "get_pesticide_information": [["topic", 11], ["product", 2]], "clarify_request": [["target", 8]], "handle_scope_limit": [["reason", 3]], "get_usage_guidance": [["topic", 6], ["delivery", 2]], "chit_chat": []}
NAMES = list(FUNCTIONS)
NEW_TOKENS = [f'<hs_{i}>' for i in range(len(NAMES))] + ['<hs_end>']
CALL = re.compile(r'^<hs_(\d+)>\(((?:[a-z_]+=-?\d+(?:, [a-z_]+=-?\d+)*)?)\)<hs_end>$')

def parse_call(text):
    m = CALL.match(text.strip())
    if not m or int(m.group(1)) >= len(NAMES):
        return None
    name = NAMES[int(m.group(1))]
    given = [p.split('=') for p in m.group(2).split(', ')] if m.group(2) else []
    if [k for k, _ in given] != [a for a, _ in FUNCTIONS[name]]:
        return None
    if any(not 1 <= int(v) <= n for (_, v), (_, n) in zip(given, FUNCTIONS[name])):
        return None
    return name, tuple(int(v) for _, v in given)

In [ ]:
# Every run saved the same tokenizer (base + 10 new tokens); check before mixing their weights.
tokenizer = AutoTokenizer.from_pretrained(DRIVE_DIR / EXPORT / 'adapter')
TOKEN_IDS = {t: tokenizer.encode(t, add_special_tokens=False) for t in NEW_TOKENS}
assert all(len(v) == 1 for v in TOKEN_IDS.values()), TOKEN_IDS
TOKEN_IDS = {t: v[0] for t, v in TOKEN_IDS.items()}
for run in RUNS:
    tok = AutoTokenizer.from_pretrained(DRIVE_DIR / run / 'adapter')
    assert all(tok.encode(t, add_special_tokens=False) == [i] for t, i in TOKEN_IDS.items()), run
END_ID = TOKEN_IDS['<hs_end>']
BASE_EOS = tokenizer.eos_token_id
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_ids(r):
    user = json.dumps({'utterance': r['user_text'], 'context': r['context']}, ensure_ascii=False, separators=(',', ':'))
    ids = tokenizer.apply_chat_template([{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}],
                                        tokenize=True, add_generation_prompt=True, enable_thinking=False)
    if isinstance(ids, Mapping):
        ids = ids['input_ids']
    if hasattr(ids, 'tolist'):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return list(ids)

def generate(m, rows, batch_size):
    m.eval()
    dev = m.get_input_embeddings().weight.device
    order = sorted(range(len(rows)), key=lambda i: len(prompt_ids(rows[i])))
    texts = [None] * len(rows)
    with torch.no_grad():
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            prompts = [prompt_ids(rows[i]) for i in idx]
            n = max(map(len, prompts))
            ids = torch.tensor([[tokenizer.pad_token_id] * (n - len(p)) + p for p in prompts], device=dev)
            mask = torch.tensor([[0] * (n - len(p)) + [1] * len(p) for p in prompts], device=dev)
            gen = m.generate(input_ids=ids, attention_mask=mask, max_new_tokens=16, do_sample=False,
                             eos_token_id=[END_ID, BASE_EOS], pad_token_id=tokenizer.pad_token_id, use_cache=True)
            for i, seq in zip(idx, gen[:, n:].tolist()):
                cut = seq.index(END_ID) + 1 if END_ID in seq else next((j for j, t in enumerate(seq) if t in (BASE_EOS, tokenizer.pad_token_id)), len(seq))
                texts[i] = tokenizer.decode(seq[:cut], skip_special_tokens=False).strip()
    return texts

def evaluate(m):
    probe = val_rows[:32]
    batch = 16 if generate(m, probe, 1) == generate(m, probe, 16) else 1
    start = time.time()
    texts = generate(m, val_rows, batch)
    return texts, dict(batch_generation=batch, seconds=round(time.time() - start))

def score(texts):
    cases = []
    for r, t in zip(val_rows, texts):
        got, gold = parse_call(t), parse_call(r['output'])
        cases.append(dict(case_id=r['case_id'], augmentation=r['augmentation'], user_text=r['user_text'],
                          expected=r['output'], actual=t, expected_function=r['function'],
                          actual_function=got[0] if got else None, format_valid=got is not None,
                          function_correct=bool(got) and got[0] == gold[0], exact_correct=t == r['output']))
    rate = lambda cs, key: round(sum(c[key] for c in cs) / max(1, len(cs)), 4)
    per = {f: dict(support=len(cs), function=rate(cs, 'function_correct'), exact=rate(cs, 'exact_correct'))
           for f in sorted({c['expected_function'] for c in cases}) for cs in [[c for c in cases if c['expected_function'] == f]]}
    return cases, dict(support=len(cases), format_valid=rate(cases, 'format_valid'), function=rate(cases, 'function_correct'),
                       exact=rate(cases, 'exact_correct'), per_function=per)

def saved_outputs(run):
    return {c['case_id']: c['actual'] for c in map(json.loads, (DRIVE_DIR / run / 'eval_cases.jsonl').read_text(encoding='utf-8').splitlines())}

def write(name, cases, metrics):
    (OUT / f'{name}_metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
    with (OUT / f'{name}_cases.jsonl').open('w', encoding='utf-8') as f:
        for c in cases:
            f.write(json.dumps(c, ensure_ascii=False) + '\n')
    print(name, json.dumps({k: metrics[k] for k in ('format_valid', 'function', 'exact')}))

MODEL_CLASS = getattr(transformers, 'Qwen3_5ForConditionalGeneration')

def load_base():
    m = MODEL_CLASS.from_pretrained(MODEL_ID, revision=MODEL_REF, device_map={'': 0}, dtype=DTYPE, attn_implementation='sdpa')
    m.resize_token_embeddings(max(len(tokenizer), m.get_input_embeddings().num_embeddings))  # same as training
    return m

def load_merged(run):
    return PeftModel.from_pretrained(load_base(), DRIVE_DIR / run / 'adapter', is_trainable=False).merge_and_unload()

def save_hf(model, folder, source):
    # Plain HF model for the RKLLM conversion; a reload must answer the same.
    check = val_rows[:64]
    expected = generate(model, check, 1)
    folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder, safe_serialization=True, max_shard_size='2GB')
    tokenizer.save_pretrained(folder)
    (folder / 'cropy_prompt.json').write_text(json.dumps(
        dict(system_prompt=SYSTEM_PROMPT, functions=FUNCTIONS, new_tokens=TOKEN_IDS, user_format='{"utterance":...,"context":...}',
             enable_thinking=False, stop_token='<hs_end>', base=MODEL_ID, revision=MODEL_REF, source=source),
        ensure_ascii=False, indent=2))
    reloaded = MODEL_CLASS.from_pretrained(folder, device_map={'': 0}, dtype=DTYPE, attn_implementation='sdpa')
    ok = generate(reloaded, check, 1) == expected
    del reloaded; gc.collect(); torch.cuda.empty_cache()
    print('저장', folder, '| 다시 불러온 결과가 같은가:', ok)
    return {'cases': len(check), 'outputs_equal': ok}

## 1. 병합 (`v2c_08b_a100_r16_e2`)
어댑터를 기본 가중치에 더해 일반 모델로 만든다. 어댑터로 평가했을 때 저장한 답(`v2c_08b_a100_r16_e2/eval_cases.jsonl`)과 얼마나 같은지 본다.
bf16에서 더하는 순서가 달라 몇 건은 달라질 수 있다.

In [ ]:
if RUN_MERGE:
    merged = load_merged(EXPORT)
    merged.config.use_cache = True
    texts, run_info = evaluate(merged)
    cases, metrics = score(texts)
    before = saved_outputs(EXPORT)
    same = sum(c['actual'] == before[c['case_id']] for c in cases)
    metrics |= dict(run=EXPORT, same_as_adapter=same, env=ENV, **run_info)
    write('merged_' + EXPORT, cases, metrics)
    print(f'어댑터 평가와 같은 답 {same}/{len(cases)}')
    if SAVE_MERGED:
        check = save_hf(merged, DRIVE_DIR / EXPORT / 'merged_hf', EXPORT)
        (OUT / 'merged_reload_check.json').write_text(json.dumps(check))
    del merged; gc.collect(); torch.cuda.empty_cache()
else:
    print('병합 건너뜀')

## 2. 평균 (모델 수프)
세 run을 각각 병합한 가중치를 평균 낸다(기본 가중치 + 세 변화량의 평균과 같다). 새 토큰 임베딩도 평균 낸다.
CPU 메모리에 float32로 더하므로 RAM이 넉넉한 런타임이 좋다(약 20GB).

In [ ]:
total = None
for run in RUNS:
    m = load_merged(run)
    sd = m.state_dict()
    if total is None:
        total = {k: v.detach().float().cpu().clone() for k, v in sd.items()}
    else:
        assert sd.keys() == total.keys(), run
        for k, v in sd.items():
            total[k] += v.detach().float().cpu()
    del m, sd; gc.collect(); torch.cuda.empty_cache()
    print('더함', run)
soup = load_base()
missing, unexpected = soup.load_state_dict({k: (v / len(RUNS)).to(DTYPE) for k, v in total.items()}, strict=False)
assert not unexpected, unexpected[:5]
assert all('lm_head' in k for k in missing), missing[:5]  # tied to the embeddings
del total; gc.collect()
soup.config.use_cache = True
if SAVE_SOUP:  # one prompt at a time, as the board will run it; the first run used batches of 16
    start = time.time()
    texts, run_info = generate(soup, val_rows, 1), dict(batch_generation=1)
    run_info['seconds'] = round(time.time() - start)
else:
    texts, run_info = evaluate(soup)
cases, metrics = score(texts)
metrics |= dict(runs=RUNS, env=ENV, **run_info)
name = SOUP_NAME + ('_b1' if run_info['batch_generation'] == 1 else '')
write(name, cases, metrics)
if SAVE_SOUP:
    (OUT / f'{name}_reload_check.json').write_text(json.dumps(save_hf(soup, DRIVE_DIR / SOUP_NAME / 'merged_hf', RUNS)))

## 요약

In [ ]:
summary = {}
for path in sorted(OUT.glob('*_metrics.json')):
    m = json.loads(path.read_text())
    summary[path.stem.replace('_metrics', '')] = {k: m[k] for k in ('format_valid', 'function', 'exact')}
(OUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
for name, v in summary.items():
    print(f'{name:24s} 형식 {v["format_valid"]:.4f} 기능 {v["function"]:.4f} 인자까지 {v["exact"]:.4f}')

In [ ]:
print('비교 기준(같은 라벨): 0.8B 단일 seed 0.9026 / 0.9112 / 0.8940 (평균 0.9026), 2B 평균 모델 0.9690')